# Gemini Research Credits — Setup Guide

Use this notebook as a concise checklist to configure Gemini on Google Cloud.

> **Credit restriction:** use the award only for **Gemini models** and other explicitly approved Google Cloud SKUs. Do **not** use third-party models (e.g. Anthropic or DeepSeek) through Model Garden / Marketplace unless separately approved.

## **Prerequisites — [Supervisor]**

### A. Confirm the billing account
1. Open **Google Cloud Console → ☰ → Billing → Credits**.
2. Confirm the Gemini promotional credit is **Available**.
3. Note the **Billing account name / ID**.

### B. Optional: let X handle the full setup
*(Requires granting X access to the Gemini credit billing account.)*

1. Go to **Billing → select the Gemini-credit billing account → Account Management / Permissions**.
2. Click **Grant access / Add principal**.
3. Enter X's Google account (email).
4. Assign **Billing Account User** → **Save**.

> **Billing Account User** lets X link projects to the billing account.
>
> Recommend: Grant **Billing Account Administrator** only when necessary.

## **1. Project setup — [Administrators]**
**(Task for the supervisor, or X if X has the required project/billing permissions.)**

### Step 1 — Create or select the project
- Click the **project selector** at the top of Google Cloud Console.
- Choose an existing project, or click **New Project** if permitted.
- After creation, select that project at the top.

### Step 2 — Verify or link the Gemini-credit billing account
- Go to **☰ → Billing**.
- Verify: **Billing account "Gemini_API_Budgets" is linked to this project**
  - If unseen, click **Link a billing account / Change billing account**.
  - Select the billing account containing the Gemini credits → **Set account / Save**.

### Step 3 — Enable Gemini API
- Go to **☰ → APIs & Services → Library**.
- Search **Gemini API** → open it → **Enable**.

### Step 4 — Create a dedicated service account
- Go to **☰ → IAM & Admin → Service Accounts → Create service account**.
- Name it `gemini-api` → **Create and continue**.
- Leave the optional role/access steps empty → **Done**.
- **Do not create/download a service-account JSON key.**

### Step 5 — Create the Gemini authorization API key
- Go to **☰ → APIs & Services → Credentials → Create credentials → API key**.
- Select/bind the `gemini-api` service account.
- Restrict the key to **Gemini API** → **Create / Save**.
- Hint: Store the key securely; **do not email/chat it or commit it to Git**.

### Step 6 — Add a student to their project as a Project Viewer

*(Optional, this help the students to check the cost in the console.)*

- Go to **☰ → IAM & Admin → IAM**.
- Click **Grant access**.
- Under **New principals**, enter the student's university/Google email address.
- Under **Assign roles**, select **Project Viewer**.
- Click **Save**.


---
## **2. To test the Gemini API (Standard Inference) - [Administrator]**

Install the Google Gen AI SDK, then enter the API key securely when prompted.

In [ ]:
%pip install -q google-genai

In [ ]:
"""
Option 1: If you are running locally (e.g., a Slurm cluster like Hábrók or Snellius)
"""

import os
from getpass import getpass

# Skippable if you already configured the key in your environment
os.environ["GEMINI_API_KEY"] = getpass("Paste Gemini API key: ")
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

In [ ]:
"""
Option 2: If you are running in Colab (e.g., a Jupyter Notebook)
"""

from google.colab import userdata

# Add the key to Colab Secrets by clicking the 🔑 icon on the left
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
"""
Running
"""

from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL = "gemini-3.8-flash"
config = types.GenerateContentConfig(
    temperature=1.0,
    top_p=0.95,
    top_k=40,
    max_output_tokens=500,
    thinking_config=types.ThinkingConfig(
        thinking_level="low", # Couldn't be shut down. Default:  medium (i.e., None). Choices: low, medium, high
        include_thoughts=True, # To enable the thinking summary in the response  (Note: we couldn't get the raw CoT chain for Gemini models)
    )
)

response = client.models.generate_content(
    model=MODEL,
    contents="Explain photosynthesis in one paragraph.",
    config=config,
)

In [ ]:
print("Response answer:")
print(response.text, '\n')
print("Thinking chain:")
for part in response.candidates[0].content.parts:
  if part.thought:
    print(part.text)

See all available Gemini models: https://ai.google.dev/gemini-api/docs/models

Find the prices: https://ai.google.dev/gemini-api/docs/pricing

> **🚨🚨🚨 Important: ALWAYS** run a smoke test with a small subset before full-corpus running to **estimate the total price!**

---
## **3. After running - [Administrator]**

- Check **Billing → Reports / Credits** to see the cost and to confirm usage is being offset by the Gemini credit.